# JARVIS RVC v2 training (Colab GPU only)

Upload `dataset/` (40 kHz mono s16 WAV, **10-30 min CLEAN single-speaker vocals**,
no music/reverb). Runtime -> Change runtime type -> **GPU (T4/A100)**.
Mac MPS training is low-quality — do NOT train locally. Target ~150-300 epochs, batch ~40.

In [ ]:
!git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git rvc
%cd rvc
!pip install -r requirements.txt
!python tools/download_models.py

In [ ]:
# preprocess: sr=40000, n_cpu=4, exp=jarvis (dataset at /content/dataset)
!python infer/modules/train/preprocess.py /content/dataset 40000 4 ./logs/jarvis True 3.0

In [ ]:
# F0 (RMVPE) + HuBERT feature extraction (v2)
!python infer/modules/train/extract/extract_f0_rmvpe.py 1 0 0 ./logs/jarvis True
!python infer/modules/train/extract_feature_print.py cuda:0 1 0 ./logs/jarvis v2

In [ ]:
# train: ~150-300 epochs (-te), batch 40 (-bs), save every 25 (-se)
!python infer/modules/train/train.py -e jarvis -sr 40k -f0 1 -bs 40 -te 250 -se 25 \
  -pg assets/pretrained_v2/f0G40k.pth -pd assets/pretrained_v2/f0D40k.pth -l 0 -c 0 -sw 1 -v v2

In [ ]:
# build the faiss index, then download the two artifacts
!python tools/train_index.py jarvis v2
from google.colab import files
files.download('logs/jarvis/added_IVF_jarvis_v2.index')
files.download('assets/weights/jarvis.pth')

## Install the result on the Mac

Copy BOTH `jarvis.pth` + `added_IVF_jarvis_v2.index` into `~/jarvis/voice_models/`
(so `rvc_model_path` / `rvc_index_path` resolve), then run JARVIS with
`JARVIS_TTS_BACKEND=melotts JARVIS_VC_BACKEND=rvc python -m jarvis`.